In [1]:
import os

In [3]:
%pwd

'd:\\Text_Summarize_Project\\Text_Summarizer_Project\\research'

In [4]:
os.chdir('../')

In [5]:
%pwd

'd:\\Text_Summarize_Project\\Text_Summarizer_Project'

In [16]:
from dataclasses import dataclass
from pathlib import Path
from text_summarize.constants import *
from text_summarize.utils.common import read_yaml,create_directories

In [17]:
@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir:Path
    data_path:Path
    tokenizer_name:Path

In [19]:
class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH):
        self.config = read_yaml(config_file_path)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_object(self)->DataTransformationConfig:
        config = self.config.data_transformation
        data_object = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_object


In [25]:
from transformers import AutoTokenizer
from datasets import load_from_disk

In [26]:
class DataTransformation:
    def __init__(self,config:DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self,example_batch):    
        input_encodings = self.tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)
        target_encodings = self.tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True,
            padding="max_length"
        )
        return {
            'input_ids':input_encodings['input_ids'],
            'attention_mask':input_encodings['attention_mask'],
            'labels':target_encodings['input_ids']
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features,batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))



In [28]:
try:
    config = ConfigurationManager()
    data_trans_obj = config.get_data_transformation_object()
    data_transformation = DataTransformation(data_trans_obj)
    data_transformation.convert()
except Exception as e:
    raise e


[2026-04-11 13:04:32,320:INFO:common:yaml file config\config.yaml loaded successfuly]
[2026-04-11 13:04:32,342:INFO:common:created directory at artifacts]


[2026-04-11 13:04:32,881:INFO:_client:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-11 13:04:32,937:INFO:_client:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-04-11 13:04:33,250:INFO:_client:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-11 13:04:33,302:INFO:_client:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-04-11 13:04:33,616:INFO:_client:HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-04-11 13:04:33

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 88932.86 examples/s]
